# Autoencoder

Excellent generic tutorial

https://www.tensorflow.org/tutorials/generative/autoencoder

# Install gwpy module to get the GW data

In [ ]:
# Uncomment only if necessary
#!pip install gwpy

# Get the GW150914 data and plot

In [ ]:
from gwpy.timeseries import TimeSeries
import pickle

fname = 'GW150914_H1.pkl'

# Load data (deserialize) from pkl file. If the opreration is not succesful download from open data repository.
try:
    with open(fname, 'rb') as handle:
        h1 = pickle.load(handle)
except:
    h1 = TimeSeries.fetch_open_data('H1', 1126259457, 1126259467)

    # Store data (serialize)
    with open(fname, 'wb') as handle:
        pickle.dump(h1, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
from gwpy.plot import Plot
%matplotlib inline

from astropy import units as u
import copy

# Create a copy of the GW data
h1b = copy.deepcopy(h1)

# Filter the data as in the GW150914 Data Analysis - Frequency filtering example
h1b = h1b.bandpass(50,250).notch(60).notch(120).crop(h1b.times[0]+1*u.s, h1b.times[-1]-0.9996*u.s)

# Plot the filtered data
plot = Plot(figsize=(12, 4))
ax = plot.gca()
ax.set_xscale('auto-gps')
ax.plot(h1b, color='gwpy:ligo-hanford', label='LIGO-Hanford')
ax.set_epoch(1126259462.427)
ax.set_ylim(-1e-21, 1e-21)
ax.set_ylabel('Strain noise')
ax.legend()
plot.show()

# Install ML modules

In [ ]:
# Uncomment only if necessary
#!pip install pandas
#!pip install scikit-learn

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, losses
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.models import Model

# Prepare training set

template length 1426 samples

half of the template contains the chirp

f_sampling 4096 Hz

In [ ]:
print("Data samples vector length: ", h1b.data.shape)

print("Sampling rate: ", h1b.sample_rate)

In [ ]:
# Auto-encoder input length
#latent_dim = 128
latent_dim = 256

# Reshape the data vector into N latent_dim vectors
# Normalize the data into ~1 order of magnitude
h1b_128 = np.reshape(h1b.data.tolist(), (-1, latent_dim)) / 10e-22
#h1b_128 = np.reshape(h1b.data.tolist(), (128, -1))

# Find minimum and maximum value in the tensor
min_val = tf.reduce_min(h1b_128)
max_val = tf.reduce_max(h1b_128)

# Normalize the tensor to [0:1] form
h1b_128 = (h1b_128 - min_val) / (max_val - min_val)

# Cast tensor data type to float32
h1b_128 = tf.cast(h1b_128, tf.float32)

# Create DataFrame fromt the tensor
df = pd.DataFrame(h1b_128)

# Create a random boolean mask of lenght df to choose which vectors will be used for training (around 80%) and which for validation (the remaining)
msk = np.random.rand(len(df)) < 0.8

# Create two DataFrames applying the mask to the input data
train = df.loc[msk]
test = df.loc[~msk]

In [ ]:
train

# Define auto-encoder model

<img src='Autoencoder_schema.png' style='width:280px;heigth:auto' />

Input vector length is the same as the output vector length.

In [ ]:
class Autoencoder(Model):
    def __init__(self, latent_dim):
        super(Autoencoder, self).__init__()
        self.latent_dim = latent_dim   
        self.encoder = tf.keras.Sequential([
          #layers.Flatten(),
          layers.Dense(latent_dim/4, activation='relu'),
        ])
        self.decoder = tf.keras.Sequential([
          layers.Dense(latent_dim, activation='sigmoid'),
          #layers.Reshape((28, 28))
        ])

    def call(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

autoencoder = Autoencoder(latent_dim)

In [ ]:
'''
class AnomalyDetector(Model):
    def __init__(self):
        super(AnomalyDetector, self).__init__()
        self.latent_dim = latent_dim
        self.encoder = tf.keras.Sequential([
          layers.Dense(latent_dim, activation="relu"),
          layers.Dense(64, activation="relu"),
          layers.Dense(32, activation="relu")])

        self.decoder = tf.keras.Sequential([
          layers.Dense(32, activation="relu"),
          layers.Dense(64, activation="relu"),
          layers.Dense(latent_dim, activation="sigmoid")])

    def call(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

autoencoder = AnomalyDetector()
'''

# Compile the model

In [ ]:
autoencoder.compile(optimizer='adam', loss=losses.MeanSquaredError())

# Train the model

In [ ]:
history = autoencoder.fit(train, train,
                epochs=200,
                shuffle=True,
                validation_data=(test, test))


# See the training statistics and the results

In [ ]:
autoencoder.encoder.summary()

In [ ]:
autoencoder.decoder.summary()

## Plot the training loss and validation loss

In [ ]:
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.legend()

## Let us check the reconstruction the auto-encoder learned

In [ ]:
# Test from the training set
#sample = train.loc[50]

# Test from the validation set
#sample = test.loc[3]

sample = df.loc[10]

plt.plot(sample, label="GW data")
plt.plot(autoencoder.call(np.expand_dims(sample, axis=1).T).numpy().T, label="AE reconstructed data")
plt.legend()

## Verify 

In [ ]:
# Normalize the GW data
h1b_np = np.expand_dims(np.array(h1b.data.tolist()), axis=1).T / 10e-22

h1b_np = (h1b_np - min_val) / (max_val - min_val)

h1b_np = tf.cast(h1b_np, tf.float32)

# Calculate number of steps
steps = h1b_np.numpy().size - latent_dim

# Prepare a vector for the result
res = np.zeros(steps)

ran = int(steps/10)-1
#ran = 40

for i in range(ran):
    inp = h1b_np[:,i*10:(i*10 + latent_dim)]       # Select the input data for the auto-encoder
    a = autoencoder.call(inp).numpy()              # Call the auto-encoder
    res[i*10] = np.power(np.sum(a[0] - inp[0]), 2) # Calculate the accuracy of the reconstruction

plt.plot(res, label="AE reconstruction error")
plt.legend()

## Double check if the auto-encoder guess was in the right moment

In [ ]:
fig, axs = plt.subplots(2, figsize=(12, 4))

# First pane
axs[0].plot(h1b.times[:-latent_dim], res, label='AE reconstruction error')
axs[0].set_xscale('auto-gps')
axs[0].legend()

# Second pane
axs[1].plot(h1b.times[:-latent_dim], np.array(h1b.data.tolist())[:-latent_dim], label='raw data', color='orange')
axs[1].set_xscale('auto-gps')
axs[1].legend()

In [ ]:
fig, axs = plt.subplots(2, figsize=(12, 6))
fig.suptitle('Zoom around event time')
startsample = 16000
stopsample = 19000

xtime = h1b.times[startsample:stopsample-latent_dim]

# First pane
axs[0].plot(xtime, res[startsample:stopsample-latent_dim], label='AE reconstruction error')
axs[0].set_xscale('auto-gps')
axs[0].legend()

# Second pane
axs[1].plot(xtime, np.array(h1b.data.tolist())[startsample:stopsample-latent_dim], label='raw data', color='orange')

forecaststart = 1850 
inp = h1b_np[:,startsample+forecaststart:(startsample+forecaststart + latent_dim)]       # Select the input data for the auto-encoder
a = np.copy(autoencoder.call(inp).numpy()[0] * 10e-22)
a.resize(xtime.shape[0])
b = np.roll(a, forecaststart)
axs[1].plot(xtime, b, label='AE reconstruction', color='green')

axs[1].set_xscale('auto-gps')
axs[1].legend()

plt.setp(axs[0], xlabel='')